## **Problem Definition**

Obtenir des détails concernant un cv

In [2]:
import json
import tiktoken

#import pandas as pd

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader
from langchain_community.vectorstores import Chroma

from dotenv.ipython import load_dotenv
import os

from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

from langchain.agents.middleware import ModelRequest,ModelResponse, wrap_model_call
from langchain.messages import HumanMessage, SystemMessage, AIMessage

from langgraph.checkpoint.memory import InMemorySaver

from langchain_experimental.tools import PythonREPLTool

# **LLM**

In [3]:
load_dotenv(override=True)

True

In [4]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# **Implementing RAG**

## **1 - Loading the PDF, Chunking**

In [5]:
pdf_file = "./pdfs/DOC-20260212-WA0048..pdf"

In [6]:
pdf_loader = PyPDFLoader(pdf_file)

In [7]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
loader = PyPDFDirectoryLoader(path = "./pdfs")

In [8]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='o200k_base',
    chunk_size=300,
    chunk_overlap=20
)

In [9]:
chunks = loader.load_and_split(text_splitter)

In [10]:
len(chunks)

14

## **2 - Vector Store - ChromaDB, Embeddings**

In [11]:
from langchain_openai import OpenAIEmbeddings
embedding_model = OpenAIEmbeddings(model='text-embedding-ada-002')

In [12]:
vectorstore = Chroma.from_documents(
    chunks,
    embedding_model,
    collection_name="rapport_ocp_V2",
    persist_directory="./store"
)

In [13]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 5}
)

In [14]:
retrieved_chunks=retriever.invoke("Quel est le nom de la personne à qui appartient ce cv?")

In [15]:
print(retrieved_chunks)

[Document(metadata={'title': 'Indeed Resume', 'page': 0, 'producer': 'Apache FOP Version 2.3', 'keywords': 'Indeed Resume', 'author': 'Indeed', 'creator': 'Indeed Resume', 'creationdate': '2026-02-10T12:06:53-06:00', 'total_pages': 3, 'page_label': '1', 'source': 'pdfs\\DOC-20260212-WA0048..pdf'}, page_content="Audes Mariana MOUSSAVOU MOUSSAVOU\nStagiaire\nCasablanca\na.moussavou@hestim.ma\n+212 778 753439\nDéterminée, sérieuse,\nautonome et consciente du\ntravail qui m'attend, je suis\npersuadée que je serai un\natout supplémentaire au sein\nde votre structure !\nInformations personnelles\nDate de naissance: 2003-07-17\nExpérience\nCertificat coursera : Exploratory Data Analysis for Machine Learning\nIBM\njanvier 2026 - aujourd'hui\n• Apprendre à récupérer et nettoyer des données\n• Faire l'analyse exploratoire des données\n• Comprendre les statistiques inférentielles et tests d'hypothèses\nBadge Coursera : Data Science Orientation\nIBM\ndécembre 2025 - aujourd'hui\n• Comprendre les b

In [16]:
print(len(retrieved_chunks))

5


## **3 - RAG Q&A**

### **Prompt Design**

In [17]:
prompt_template = """
Answer the following question based only on provided context
The context is about DOC-20260212-WA0048
The context is delimited by <context> tag
The user question is delimited by <question> tag
If the answer is not found in the context, answer : JE NE SAIS PAS
<context>
 {context}
</context>
<question>
 {question}
</question>
"""

### **Retrieving the Relevant Documents**

In [18]:
user_input = "Quel est le nom de la personne à qui appartient ce cv?"

In [19]:
relevant_document_chunks = retriever.invoke(user_input)
context_list = [d.page_content for d in relevant_document_chunks]
context_for_query = ". ".join(context_list)

In [20]:
context_for_query

"Audes Mariana MOUSSAVOU MOUSSAVOU\nStagiaire\nCasablanca\na.moussavou@hestim.ma\n+212 778 753439\nDéterminée, sérieuse,\nautonome et consciente du\ntravail qui m'attend, je suis\npersuadée que je serai un\natout supplémentaire au sein\nde votre structure !\nInformations personnelles\nDate de naissance: 2003-07-17\nExpérience\nCertificat coursera : Exploratory Data Analysis for Machine Learning\nIBM\njanvier 2026 - aujourd'hui\n• Apprendre à récupérer et nettoyer des données\n• Faire l'analyse exploratoire des données\n• Comprendre les statistiques inférentielles et tests d'hypothèses\nBadge Coursera : Data Science Orientation\nIBM\ndécembre 2025 - aujourd'hui\n• Comprendre les bases de la science des données\n• Comprendre la relation entre la science des données et l’IA\nBadge Networking Basics\nCisco Networking Academy\ndécembre 2025 - aujourd'hui\n• Connaître les bases des réseaux informatiques\n• Connaître les différentes types de réseau\n• Connaitre les équipements réseaux et leur

In [21]:
# Here the length is 10 because, earlier we have declared k=10
len(relevant_document_chunks)

5

In [22]:
prompt = prompt_template.format(context=context_for_query, question=user_input)

In [23]:
print(prompt)


Answer the following question based only on provided context
The context is about DOC-20260212-WA0048
The context is delimited by <context> tag
The user question is delimited by <question> tag
If the answer is not found in the context, answer : JE NE SAIS PAS
<context>
 Audes Mariana MOUSSAVOU MOUSSAVOU
Stagiaire
Casablanca
a.moussavou@hestim.ma
+212 778 753439
Déterminée, sérieuse,
autonome et consciente du
travail qui m'attend, je suis
persuadée que je serai un
atout supplémentaire au sein
de votre structure !
Informations personnelles
Date de naissance: 2003-07-17
Expérience
Certificat coursera : Exploratory Data Analysis for Machine Learning
IBM
janvier 2026 - aujourd'hui
• Apprendre à récupérer et nettoyer des données
• Faire l'analyse exploratoire des données
• Comprendre les statistiques inférentielles et tests d'hypothèses
Badge Coursera : Data Science Orientation
IBM
décembre 2025 - aujourd'hui
• Comprendre les bases de la science des données
• Comprendre la relation entre la

In [24]:
resp = llm.invoke(prompt)

In [25]:
from IPython.display import Markdown

In [26]:
print(display(Markdown(resp.content)))

Audes Mariana MOUSSAVOU MOUSSAVOU

None


### **Defining the RAG function for response**




In [27]:
def RAG(query, llm=llm, prompt_template=prompt_template):
    context_docs = retriever.invoke(query)
    context_list = [d.page_content for d in context_docs]
    context_for_query = ". ".join(context_list)
    prompt = prompt_template.format(context=context_for_query, question=query)
    resp=llm.invoke(prompt)
    return resp.content

In [28]:
response = RAG("Est-ce un profil de data scientiste?")
print(display(Markdown(response)))

Oui, le profil présente des compétences et des expériences pertinentes pour un data scientiste, notamment des certificats en analyse de données, en science des données et en programmation.

None


In [29]:
user_input = "j'ai fain et je veux manger quelque chose"
output = RAG(user_input)
print(output)

JE NE SAIS PAS


In [30]:
response = RAG("Est-ce un profil d'une administratrice réseaux junior?")
print(display(Markdown(response)))

JE NE SAIS PAS

None


In [31]:
user_input ="Est-ce un profil de data scientiste?"

In [32]:
relevant_document_chunks = retriever.invoke(user_input)
context_list = [d.page_content for d in relevant_document_chunks]
context_for_query = ". ".join(context_list)

In [33]:
user_message_template = """
 ###Question
 {question}
 ###Context
 {context}
 ###Answer
 {answer}
"""

In [34]:
# Default answer for an RAG query
answer = RAG(user_input)
print(display(Markdown(answer)))

Oui, le profil présente des compétences et des expériences pertinentes pour un data scientiste, notamment des certificats en analyse de données, en science des données et en programmation.

None


In [35]:
groundness_checker = ChatOpenAI(
    model="gpt-4o",
    temperature=0
)

In [36]:
def evaluate(system_message,user_message_template, question, model=groundness_checker):
    retrieved_chunks=retriever.invoke(question)
    context_list = [d.page_content for d in retrieved_chunks]
    context = ". ".join(context_list)
    answer = RAG(question)
    prompt = f"""
     {system_message}\n
     USER:
     {user_message_template.format(question=question, context=context, answer=answer)}
    """
    juge_response= model.invoke(prompt)
    return juge_response.content


# Agentic RAG

In [37]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model = OpenAIEmbeddings(model="text-embedding-3-large")

In [51]:
# -------------------------------
# RAG as Tool - Exemple Complet
# -------------------------------

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.tools import create_retriever_tool
from langchain_core.messages import HumanMessage
from IPython.display import display, Markdown

# Étape 1 : Créer le modèle d'embedding
embedding_model = OpenAIEmbeddings(model="text-embedding-3-large")

# Étape 2 : Corpus de textes
texts = [
    "Je m'appelle Audes Mariana, Je suis étudiante en Informatique et Intelligence artificielle.",
    "J'ai obtenu en 2015, mon Doctorat et titre de Docteur d'état en informatique",
    "Je suis etudiante et je vis à la ville de casablanca",
    "Je suis Marocain",
]

# Étape 3 : Créer le vecteur store FAISS
vector_store = FAISS.from_texts(texts=texts, embedding=embedding_model)

# Étape 4 : Exemple de recherche de similarité
results = vector_store.similarity_search("Nom, prénom et affiliation", k=2)
print("Résultats de similarité :")
print(results[0].page_content)
print(results[1].page_content)

# Étape 5 : Créer un outil de retrieval
retrieval = vector_store.as_retriever(kwargs={'k': 5})
retrieval_tool = create_retriever_tool(
    retriever=retrieval,
    name="kb_search",
    description="Search information about me"
)

# Étape 6 : Créer un agent avec GPT et l’outil
search_agent = create_agent(
    model="openai/gpt-5.2",
    system_prompt="Search information about me",
    tools=[retrieval_tool]
)


# Étape 7 : Interroger l’agent
query = "Nom et Prénom, Affiliation et diplômes"
result = search_agent.invoke({'messages': [HumanMessage(query)]})

# Afficher le résultat formaté
print("\nRésultat de l'agent :")
display(Markdown(result['messages'][-1].content))


ImportError: Could not import faiss python package. Please install it with `pip install faiss-gpu` (for CUDA supported GPU) or `pip install faiss-cpu` (depending on Python version).